# Dependencies

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr poppler-utils

!pip -q install -U \
  "fastcoref==2.1.6" \
  "spacy>=3.7,<3.9" \
  "transformers==4.38.2" \
  "tokenizers==0.15.2" \
  "accelerate>=0.33.0" \
  "sentence-transformers==2.6.1"

!pip -q install ragas datasets langchain-openai

!pip -q install -U pypdf sentence-transformers hnswlib rank-bm25 pytesseract pdf2image pillow "Pillow==11.0"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import re
import numpy as np
import torch
import os
import time
from tqdm.auto import tqdm

from pypdf import PdfReader

from pdf2image import convert_from_path
import pytesseract
from PIL import Image

from google.colab import drive
from pathlib import Path
import json

from sentence_transformers import SentenceTransformer, util, CrossEncoder
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, pipeline
from rank_bm25 import BM25Okapi

import hnswlib

# GDrive Setup

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR      = Path("/content/drive/MyDrive/PGD_Colab")
RAW_DOCS_DIR  = BASE_DIR / "raw_docs"

# New Jurisprudence structure
JURIS_RAW_DIR  = RAW_DOCS_DIR / "Jurisprudence" / "Raw"
JURIS_SUM_DIR  = RAW_DOCS_DIR / "Jurisprudence" / "Structured"

# Cache root
CACHE_ROOT    = BASE_DIR / "cache" / "RA9165_RAG"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

ALLOWED_EXTS = {".pdf", ".txt"}

In [ ]:
def list_docs(folder: Path, exts=ALLOWED_EXTS, exclude_substrings=None):
    if not folder.exists():
        print(f"⚠️ Missing folder: {folder}")
        return []
    paths = []
    for ext in exts:
        for p in folder.rglob(f"*{ext}"):
            p_str = str(p)
            p_norm = p_str.replace("\\", "/")
            if exclude_substrings and any(excl in p_norm for excl in exclude_substrings):
                continue
            paths.append(p_str)
    return sorted(set(paths))

In [ ]:
corpus_files = {}

# Exclude jurisprudence folders from statutes corpus
corpus_files["statutes_and_guidelines"] = list_docs(
    RAW_DOCS_DIR,
    exclude_substrings=["Jurisprudence/Raw", "Jurisprudence/Structured"]
)

corpus_files["jurisprudence"] = list_docs(JURIS_RAW_DIR)
corpus_files["jurisprudence_structured"] = list_docs(JURIS_SUM_DIR)

total = sum(len(v) for v in corpus_files.values())

for name, files in corpus_files.items():
    print(f"[{name}] -> {len(files)} docs")

print("TOTAL docs:", total)

[statutes_and_guidelines] -> 5 docs
[jurisprudence] -> 5 docs
[jurisprudence_structured] -> 10 docs
TOTAL docs: 20


In [ ]:
def get_cache_paths(corpus_name: str):
    cache_dir = CACHE_ROOT / corpus_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    return {
        "CACHE_DIR": cache_dir,
        "CHUNKS_PATH": cache_dir / "chunks.cleaned.jsonl",
        "EMBED_PATH": cache_dir / "embeddings.npy",
        "HNSW_INDEX_PATH": cache_dir / "hnsw.index",
        "HNSW_META_PATH": cache_dir / "hnsw_meta.json",
        "BM25_PATH": cache_dir / "bm25_tokens.jsonl",
        "MANIFEST_PATH": cache_dir / "manifest.json",
    }

def write_json(path, obj):
    with open(path,"w",encoding="utf-8") as f:
        json.dump(obj,f,ensure_ascii=False,indent=2)

def read_json(path):
    with open(path,"r",encoding="utf-8") as f:
        return json.load(f)

def write_jsonl(path, rows):
    with open(path,"w",encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r,ensure_ascii=False)+"\n")

def read_jsonl(path):
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def cache_exists(paths: dict):
    needed = [paths["CHUNKS_PATH"], paths["EMBED_PATH"], paths["HNSW_INDEX_PATH"], paths["HNSW_META_PATH"], paths["BM25_PATH"]]
    missing = [p.name for p in needed if not p.exists()]
    if missing:
        print(f"❌ Cache missing in {paths['CACHE_DIR'].name}: {missing}")
        return False
    print(f"✅ Cache exists for {paths['CACHE_DIR'].name}")
    return True

def load_cache(paths: dict):
    chunks = read_jsonl(paths["CHUNKS_PATH"])
    embeddings = np.load(paths["EMBED_PATH"]).astype(np.float32)

    meta = read_json(paths["HNSW_META_PATH"])
    index = hnswlib.Index(space=meta["space"], dim=int(meta["dim"]))
    index.load_index(str(paths["HNSW_INDEX_PATH"]))
    index.set_ef(meta.get("ef", 50))

    bm25_rows = read_jsonl(paths["BM25_PATH"])
    bm25_corpus = [r["toks"] for r in bm25_rows]
    bm25 = BM25Okapi(bm25_corpus)

    return chunks, embeddings, index, bm25

def save_cache(paths: dict, chunks, embeddings, index, bm25_corpus):
    write_jsonl(paths["CHUNKS_PATH"], chunks)
    np.save(paths["EMBED_PATH"], embeddings)

    index.save_index(str(paths["HNSW_INDEX_PATH"]))
    write_json(paths["HNSW_META_PATH"], {"space":"cosine","dim":int(embeddings.shape[1]),"ef":50})
    write_jsonl(paths["BM25_PATH"], [{"toks": toks} for toks in bm25_corpus])

    write_json(paths["MANIFEST_PATH"], {
        "created_utc": int(time.time()),
        "num_chunks": len(chunks),
        "embedding_dim": int(embeddings.shape[1]),
    })
    print("✅ Saved cache:", paths["CACHE_DIR"])

# Preprocessing

In [ ]:
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
RERANK_MODEL = "BAAI/bge-reranker-large"
COREF_MODEL = "biu-nlp/lingmess-coref"
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
# ----------------------------
# Tokenizer for TOKEN-based chunking
# ----------------------------
CHUNK_TOKENIZER = AutoTokenizer.from_pretrained(EMBED_MODEL, use_fast=True)

# ----------------------------
# PDF/TXT extraction
# ----------------------------
def extract_pdf_pages(path: str, ocr_if_empty=True, ocr_dpi=250, min_chars_for_text=40):
    """
    Extract text per page using pypdf; OCR only pages that are empty/too short.
    """
    reader = PdfReader(path)
    extracted = []
    ocr_idxs = []

    for i, page in enumerate(reader.pages):
        text = ""
        try:
            text = (page.extract_text() or "").strip()
        except Exception as e:
            print(f"⚠️ extract_text failed: {Path(path).name} page {i+1} -> {type(e).__name__}: {e}")
            text = ""

        if ocr_if_empty and len(text) < min_chars_for_text:
            ocr_idxs.append(i)

        extracted.append(text)

    if ocr_if_empty and ocr_idxs:
        try:
            images = convert_from_path(path, dpi=ocr_dpi)
            for i in ocr_idxs:
                ocr_text = (pytesseract.image_to_string(images[i]) or "").strip()
                if len(ocr_text) > len(extracted[i]):
                    extracted[i] = ocr_text
        except Exception as e:
            print(f"⚠️ OCR failed for {Path(path).name} -> {type(e).__name__}: {e}")

    return [{"source": path, "page": i + 1, "text": t} for i, t in enumerate(extracted)]


def extract_source_pages(path: str):
    """
    Returns list of {"source":..., "page":..., "text":...}
    """
    p = (path or "").lower()
    if p.endswith(".pdf"):
        return extract_pdf_pages(path)
    if p.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        return [{"source": path, "page": 1, "text": text}]
    return []

# ----------------------------
# Text cleanup helpers
# ----------------------------
_WS_RE = re.compile(r"[ \t]+")
_NL3_RE = re.compile(r"\n{3,}")

def normalize_text_keep_newlines(t: str) -> str:
    t = (t or "").replace("\r", "\n")
    t = _WS_RE.sub(" ", t)
    t = _NL3_RE.sub("\n\n", t)
    return t.strip()

def dehyphenate_ocr(t: str) -> str:
    # "exam-\nple" -> "example"
    return re.sub(r"(\w)-\n(\w)", r"\1\2", t or "")

# ----------------------------
# TOKEN-based chunkers
# ----------------------------
def chunk_tokens_sliding_window(text: str, tokenizer, max_tokens=420, overlap_tokens=80):
    """
    Token-based sliding window chunking.
    Returns list[str] chunks.
    """
    text = normalize_text_keep_newlines(dehyphenate_ocr(text))
    if not text:
        return []

    ids = tokenizer.encode(text, add_special_tokens=False)
    if not ids:
        return []

    if len(ids) <= max_tokens:
        return [tokenizer.decode(ids)]

    chunks = []
    start = 0
    n = len(ids)

    while start < n:
        end = min(n, start + max_tokens)
        piece_ids = ids[start:end]
        chunk_text = tokenizer.decode(piece_ids, skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)

        if end == n:
            break

        start = max(0, end - overlap_tokens)

    return chunks


def chunk_tokens_paragraph_first(text: str, tokenizer, max_tokens=420, overlap_tokens=80):
    """
    Paragraph-first token chunking:
    - split by blank lines
    - keep whole paragraphs if they fit
    - if a paragraph is too long, split it using token sliding window
    - pack multiple small paragraphs into a chunk up to max_tokens
    """
    text = normalize_text_keep_newlines(dehyphenate_ocr(text))
    if not text:
        return []

    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if not paras:
        return []

    out = []
    buffer = []
    buffer_ids = []

    sep_ids = tokenizer.encode("\n\n", add_special_tokens=False)

    def flush():
        nonlocal buffer, buffer_ids
        if buffer:
            out.append("\n\n".join(buffer).strip())
            buffer = []
            buffer_ids = []

    for p in paras:
        p_ids = tokenizer.encode(p, add_special_tokens=False)

        # paragraph too big -> flush and split paragraph
        if len(p_ids) > max_tokens:
            flush()
            out.extend(chunk_tokens_sliding_window(p, tokenizer, max_tokens=max_tokens, overlap_tokens=overlap_tokens))
            continue

        if not buffer:
            buffer = [p]
            buffer_ids = p_ids
            continue

        # try pack into buffer
        if len(buffer_ids) + len(sep_ids) + len(p_ids) <= max_tokens:
            buffer.append(p)
            buffer_ids = buffer_ids + sep_ids + p_ids
        else:
            flush()
            buffer = [p]
            buffer_ids = p_ids

    flush()
    return out


def chunk_tokens_summary_txt(text: str, tokenizer, max_tokens=520, overlap_tokens=80):
    """
    For your structured jurisprudence summaries (TXT):
    paragraph-first with a slightly larger token budget.
    """
    return chunk_tokens_paragraph_first(text, tokenizer, max_tokens=max_tokens, overlap_tokens=overlap_tokens)

# ----------------------------
# Corpus-aware build
# ----------------------------
def bm25_tokenize(text: str):
    return re.findall(r"\b\w+\b", (text or "").lower())

# ----------------------------
# Embedding Model Initialization
# ----------------------------
embedding_model = SentenceTransformer(EMBED_MODEL)

def build_corpus(corpus_name: str, file_paths: list):
    """
    Build chunks, BM25 corpus, embeddings, and HNSW index for a corpus.
    Uses TOKEN-based chunking depending on corpus_name.
    """
    t0 = time.time()
    tok = CHUNK_TOKENIZER

    # Choose TOKEN chunking strategy per corpus
    if corpus_name == "jurisprudence":
        # raw PDFs: straight token windows
        chunker = lambda txt: chunk_tokens_sliding_window(txt, tok, max_tokens=420, overlap_tokens=80)
    elif corpus_name == "jurisprudence_structured":
        # TXT summaries: paragraph-first token chunks (bigger budget)
        chunker = lambda txt: chunk_tokens_summary_txt(txt, tok, max_tokens=450, overlap_tokens=80)
    else:
        # statutes_and_guidelines: paragraph-first token chunks
        chunker = lambda txt: chunk_tokens_paragraph_first(txt, tok, max_tokens=360, overlap_tokens=80)

    print(f"\n=== BUILD CORPUS: {corpus_name} ===")
    print(f"Files: {len(file_paths)}")

    # A) Extract pages and chunk per file
    chunks = []
    for fp in tqdm(file_paths, desc=f"[{corpus_name}] Extract+Chunk", unit="file"):
        pages = extract_source_pages(fp)
        if not pages:
            continue

        chunk_counter = 0
        fname = Path(fp).name

        for pg in pages:
            pg_text = (pg.get("text") or "").strip()
            if not pg_text:
                continue

            pieces = chunker(pg_text)
            for piece in pieces:
                if not piece.strip():
                    continue
                cid = f"{fname}#p{int(pg.get('page',1))}#c{chunk_counter:06d}"
                chunk_counter += 1
                chunks.append({
                    "chunk_id": cid,
                    "source": fp,
                    "page": int(pg.get("page", 1)),
                    "text": piece
                })

    print(f"✔ Chunks: {len(chunks):,}")

    # B) BM25
    bm25_corpus = [bm25_tokenize(c["text"]) for c in tqdm(chunks, desc=f"[{corpus_name}] BM25 tokenize", unit="chunk")]
    bm25 = BM25Okapi(bm25_corpus)

    # C) Embeddings
    texts = [c["text"] for c in chunks]
    embeddings = embedding_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    # D) HNSW
    dim = int(embeddings.shape[1])
    index = hnswlib.Index(space="cosine", dim=dim)
    index.init_index(max_elements=len(embeddings) + 5000, ef_construction=400, M=16)
    ids = np.arange(len(embeddings))
    index.add_items(embeddings, ids)
    index.set_ef(400)

    print(f"✅ BUILD DONE: {corpus_name} in {time.time()-t0:.2f}s")
    return chunks, embeddings, index, bm25, bm25_corpus

# ----------------------------
# Build or load all corpora
# ----------------------------
stores = {}

for corpus_name, files in corpus_files.items():
    paths = get_cache_paths(corpus_name)

    if cache_exists(paths):
        chunks, embeddings, index, bm25 = load_cache(paths)
    else:
        chunks, embeddings, index, bm25, bm25_corpus = build_corpus(corpus_name, files)
        save_cache(paths, chunks, embeddings, index, bm25_corpus)

    stores[corpus_name] = {
        "paths": paths,
        "chunks": chunks,
        "embeddings": embeddings,
        "index": index,
        "bm25": bm25
    }

print("\n✅ Loaded stores:", {k: len(v["chunks"]) for k, v in stores.items()})

# ----------------------------
# Active corpus (change anytime)
# ----------------------------
ACTIVE_CORPUS = "statutes_and_guidelines"  # or "jurisprudence" or "jurisprudence_structured"
active_store = stores[ACTIVE_CORPUS]

chunks = active_store["chunks"]
embeddings = active_store["embeddings"]
index = active_store["index"]
bm25 = active_store["bm25"]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cache exists for statutes_and_guidelines
✅ Cache exists for jurisprudence
✅ Cache exists for jurisprudence_structured

✅ Loaded stores: {'statutes_and_guidelines': 306, 'jurisprudence': 206, 'jurisprudence_structured': 10}


In [ ]:
# ----------------------------
# SIMPLE CORPUS DIAGNOSTICS
# ----------------------------
print("ACTIVE CORPUS:", ACTIVE_CORPUS)
print()

# Number of chunks
print("Chunks:", len(chunks))

# Chunk size example
sample_tokens = len(CHUNK_TOKENIZER.encode(chunks[0]["text"], add_special_tokens=False))
print("Example chunk token size:", sample_tokens)

# Embeddings
print("\nEmbeddings shape:", embeddings.shape)
print("Embedding dimension:", embeddings.shape[1])
print("Embedding dtype:", embeddings.dtype)

# Approx embedding memory
embedding_mem = embeddings.nbytes / (1024**2)
print("Embeddings memory:", round(embedding_mem,2), "MB")

# HNSW index
print("\nHNSW index elements:", index.get_current_count())

# BM25
print("\nBM25 documents:", len(bm25.doc_freqs))

# Cache file sizes
paths = active_store["paths"]

def file_size(path):
    if path.exists():
        return round(path.stat().st_size / (1024**2), 2)
    return 0

print("\nCache sizes:")
print("chunks.cleaned.jsonl:", file_size(paths["CHUNKS_PATH"]), "MB")
print("embeddings.npy:", file_size(paths["EMBED_PATH"]), "MB")
print("hnsw.index:", file_size(paths["HNSW_INDEX_PATH"]), "MB")
print("bm25_tokens.jsonl:", file_size(paths["BM25_PATH"]), "MB")

ACTIVE CORPUS: jurisprudence

Chunks: 206
Example chunk token size: 420

Embeddings shape: (206, 768)
Embedding dimension: 768
Embedding dtype: float32
Embeddings memory: 0.6 MB

HNSW index elements: 206

BM25 documents: 206

Cache sizes:
chunks.cleaned.jsonl: 0.4 MB
embeddings.npy: 0.6 MB
hnsw.index: 0.63 MB
bm25_tokens.jsonl: 0.52 MB


# Model Initialization

## General

In [ ]:
# ============================================================
# 1) Coreference Model (LingMess)
# ============================================================

# Force eager attention (fix Longformer SDPA crash)
_old_cfg = AutoConfig.from_pretrained

def _patched_cfg(*args, **kwargs):
    cfg = _old_cfg(*args, **kwargs)
    if hasattr(cfg, "attn_implementation"):
        cfg.attn_implementation = "eager"
    if hasattr(cfg, "_attn_implementation"):
        cfg._attn_implementation = "eager"
    return cfg

AutoConfig.from_pretrained = _patched_cfg

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

from fastcoref.coref_models.modeling_lingmess import LingMessModel
if not hasattr(LingMessModel, "all_tied_weights_keys"):
    LingMessModel.all_tied_weights_keys = {}

from fastcoref import LingMessCoref

coref_model = LingMessCoref(
    model_name_or_path=COREF_MODEL,
    device="cuda:0" if torch.cuda.is_available() else "cpu",
)

print("✅ LingMessCoref loaded")

# ============================================================
# 2) Qwen LLM
# ============================================================

print("\n🔹 Loading Qwen LLM...")

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
chat_tokenizer = llm_tokenizer

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=llm_tokenizer,
    do_sample=False,
    temperature=0.0,
    max_new_tokens=250,
    return_full_text=False
)

print("✅ Qwen loaded")

# ============================================================
# 3) Reranker (BGE)
# ============================================================

print("\n🔹 Loading reranker...")

reranker = CrossEncoder(
    RERANK_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu",
    max_length=512
)

print("✅ Reranker loaded")

print("\n✅ ALL MODELS INITIALIZED SUCCESSFULLY")


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker loaded

✅ ALL MODELS INITIALIZED SUCCESSFULLY


## Sparse

In [ ]:
def retrieve_bm25(query: str, k=25, allowed_sources=None):
    q_tokens = bm25_tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argsort(scores)[::-1][:max(k*5, k)]  # pull extra then filter

    results = []
    for i in top_idx:
        i = int(i)
        h = {**chunks[i], "score": float(scores[i]), "idx": i}  # ✅ add idx
        results.append(h)

    return results[:k]

## Dense

In [ ]:
def dense_query_for_bge_v15(q: str, short_tokens: int = 8) -> str:
    """
    For BAAI/bge-base-en-v1.5:
    - Short queries benefit from the instruction prefix.
    - Longer / already-specific queries often do better without it.
    """
    q = (q or "").strip()
    if not q:
        return q

    # simple token count (split on whitespace)
    if len(q.split()) <= int(short_tokens):
        return "Represent this sentence for searching relevant passages: " + q

    return q

In [ ]:
def retrieve_hnsw(query: str, k=25, allowed_sources=None):
    query2 = dense_query_for_bge_v15(query, short_tokens=8)
    q_emb = embedding_model.encode(
        [query2],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    # How many items are actually in the index?
    n = int(index.get_current_count())
    if n <= 0:
        return []

    # Pull extra candidates but keep it sane
    pull_k = max(int(k * 8), int(k))
    pull_k = min(pull_k, n)  # never ask for more than exists

    # IMPORTANT: ef must be >= k for knn_query to work reliably
    # Set ef to at least pull_k (or cap it if you want)
    try:
        index.set_ef(max(50, pull_k))
    except Exception:
        pass

    labels, distances = index.knn_query(q_emb, k=pull_k)

    results = []
    for idx, dist in zip(labels[0], distances[0]):
        i = int(idx)
        score = 1.0 - float(dist)
        results.append({**chunks[i], "score": score, "idx": i})

    # Return top-k (not pull_k)
    return results[:k]

## Hybrid

### RRF

In [ ]:
def rrf_fusion(dense_hits, sparse_hits, rrf_k=60):
    fused_scores = {}

    def add_list(hits):
        for rank, h in enumerate(hits):
            cid = int(h["idx"])  # ✅ fuse by internal index
            fused_scores[cid] = fused_scores.get(cid, 0.0) + 1.0 / (rrf_k + rank + 1)

    add_list(dense_hits)
    add_list(sparse_hits)

    fused = []
    for cid, score in fused_scores.items():
        fused.append({**chunks[cid], "score": float(score), "idx": int(cid)})

    fused.sort(key=lambda x: float(x.get("score", 0.0) or 0.0), reverse=True)
    return fused

In [ ]:
def rrf_fusion_weighted(dense_hits, sparse_hits, rrf_k=60, w_dense=0.8, w_sparse=1.2):
    fused_scores = {}

    def add_list(hits, w):
        for rank, h in enumerate(hits):
            cid = int(h["idx"])  # ✅ fuse by internal embeddings/chunks index
            fused_scores[cid] = fused_scores.get(cid, 0.0) + w * (1.0 / (rrf_k + rank + 1))

    add_list(dense_hits,  w_dense)
    add_list(sparse_hits, w_sparse)

    fused = [{**chunks[cid], "score": float(score), "idx": int(cid)}
             for cid, score in fused_scores.items()]

    fused.sort(key=lambda x: float(x.get("score", 0.0) or 0.0), reverse=True)
    return fused

### Rerank

In [ ]:
def rerank_hits(query: str, hits, top_n=30):
    top_n = min(top_n, len(hits))
    pairs = [(query, h["text"]) for h in hits[:top_n]]
    scores = reranker.predict(pairs)

    reranked = []
    for h, s in zip(hits[:top_n], scores):
        h2 = h.copy()
        h2["rerank_score"] = float(s)
        reranked.append(h2)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked


### Diverse

In [ ]:
def diversify_by_page(hits, max_per_page=6):
    """
    Keep at most `max_per_page` hits per (source,page) to reduce duplicates.
    """
    out = []
    seen = {}
    for h in hits:
        key = (h["source"], h["page"])
        if seen.get(key, 0) < max_per_page:
            out.append(h)
            seen[key] = seen.get(key, 0) + 1
    return out


### MMR

In [ ]:
def select_diverse_hits(hits, top_k=5, lambda_mult=0.7):
    """
    MMR-style selection:
    - lambda_mult close to 1.0 => prioritize relevance
    - lambda_mult closer to 0.0 => prioritize diversity
    """
    if not hits:
        return []

    selected = []
    selected_ids = []

    # (optional) not used, but keep if you want
    cand_ids = [h["idx"] for h in hits]

    selected.append(hits[0])
    selected_ids.append(hits[0]["idx"])

    while len(selected) < top_k and len(selected) < len(hits):
        best_i = None
        best_val = -1e9

        for i, h in enumerate(hits):
            cid = h["idx"]
            if cid in selected_ids:
                continue

            rel = float(h.get("score", 0.0) or 0.0)

            v = embeddings[cid]
            sims = [float(np.dot(v, embeddings[sid])) for sid in selected_ids]
            max_sim = max(sims) if sims else 0.0

            mmr = lambda_mult * rel - (1 - lambda_mult) * max_sim
            if mmr > best_val:
                best_val = mmr
                best_i = i

        if best_i is None:
            break

        chosen = hits[best_i]
        selected.append(chosen)
        selected_ids.append(chosen["idx"])

    return selected


In [ ]:
def select_diverse_hits_v2(hits, top_k=10, lambda_mult=0.7):
    if not hits:
        return []

    selected = [hits[0]]
    selected_ids = {hits[0]["idx"]}

    while len(selected) < top_k and len(selected) < len(hits):
        best = None
        best_val = -1e9

        for h in hits:
            cid = h["idx"]
            if cid in selected_ids:
                continue

            rel = float(h.get("rerank_score", h.get("score", 0.0)) or 0.0)

            v = embeddings[cid]
            sims = [float(np.dot(v, embeddings[s["idx"]])) for s in selected]
            max_sim = max(sims) if sims else 0.0

            mmr = lambda_mult * rel - (1 - lambda_mult) * max_sim
            if mmr > best_val:
                best_val = mmr
                best = h

        if best is None:
            break

        selected.append(best)
        selected_ids.add(best["idx"])

    return selected

# Other Helpers

## Routing

In [ ]:
def route_corpus(query: str) -> str:
    q = (query or "").lower()

    # Jurisprudence detection FIRST
    juris_keywords = [
        "people v", "vs.", "g.r.", "gr no", "ruling", "case", "cases",
        "jurisprudence", "acquit", "acquitted", "convict", "convicted",
        "conviction", "court held", "doctrine",
        "lim", "consada", "sarabia", "padollo", "asaytuno"
    ]

    if any(x in q for x in juris_keywords):
        return "jurisprudence"   # ✅ raw PDF corpus

    # Statutes
    statutes_keywords = [
        "schedule i", "schedule ii", "section", "article", "irr",
        "penalt", "penalty", "ra 9165", "ra 10640", "ddb", "pdea"
    ]

    if any(x in q for x in statutes_keywords):
        return "statutes_and_guidelines"

    return "statutes_and_guidelines"

Suggest Queries

In [ ]:
def suggest_queries(user_query: str):
    base = user_query.strip()
    suggestions = [
        f'Define key terms in: "{base}"',
        f'What section/provision covers: "{base}"',
        f'Penalties and sanctions for: "{base}"',
        f'Elements/requirements for: "{base}"'
    ]
    return suggestions

Has Uncited & Regenerate Strictly

In [ ]:
import re

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def has_uncited_sentences(answer_text: str) -> bool:
    """
    Returns True if any sentence looks like it makes a claim but has no [n] citation.
    This is a strict faithfulness gate.
    """
    t = (answer_text or "").strip()
    if not t:
        return False

    # Split into sentences
    sents = [s.strip() for s in _SENT_SPLIT.split(t) if s.strip()]
    if not sents:
        sents = [t]

    for s in sents:
        # Allow headings like "RULE:" or "APPLICATION:" without citations
        if re.match(r"^[A-Z][A-Z\s]{2,}:\s*$", s):
            continue

        # Must contain at least one citation marker [n]
        if not re.search(r"\[\d+\]", s):
            return True

    return False


def regenerate_strictly(mode: str, context: str, question: str) -> str:
    """
    Force a strict rewrite where every sentence ends with a citation [n].
    """
    strict_addon = (
        "\n\nSTRICT RULES:\n"
        "- Every sentence MUST contain at least one citation like [1].\n"
        "- Do NOT write any sentence without a citation.\n"
        "- If context is insufficient, use the IDK format.\n"
    )

    onepass_msgs = [
        {"role": "system", "content": build_system_prompt()},
        {"role": "user", "content": build_onepass_prompt(mode, context, question) + strict_addon}
    ]

    onepass_prompt = chat_tokenizer.apply_chat_template(
        onepass_msgs, add_generation_prompt=True, tokenize=False
    )
    out = gen(onepass_prompt)[0]["generated_text"]
    return normalize_onepass_output((out or "").strip())

## Response

In [ ]:
CONTROL_TAGS = {
    "summary":  "SUMMARY",
    "tldr":     "SUMMARY",
    "lengthen": "DETAILED",
    "detailed": "DETAILED",
    "list":     "LIST",
    "bullets":  "LIST",
    "steps":    "STEPS",
    "procedure":"STEPS",
    "compare":  "COMPARE",
    "vs":       "COMPARE",
    "scenario": "SCENARIO",
    "yesno":    "YESNO",
    "cite":     "CITATIONS",
    "quotes":   "QUOTES",
}

def parse_control_block(user_query: str):
    """
    Returns (mode_override, cleaned_query).
    mode_override is one of:
      SUMMARY, DETAILED, LIST, STEPS, COMPARE, SCENARIO, YESNO, CITATIONS, QUOTES
    """
    q = user_query.strip()
    if not q.startswith("/"):
        return None, q

    # Allow multiple tags like "/summary /cite question..."
    parts = q.split()
    mode = None
    consumed = 0
    for p in parts:
        if not p.startswith("/"):
            break
        tag = p[1:].lower().strip()
        if tag in CONTROL_TAGS:
            mode = CONTROL_TAGS[tag]  # last one wins
            consumed += 1
        else:
            break

    cleaned = " ".join(parts[consumed:]).strip()
    if not cleaned:
        cleaned = user_query.strip()  # fallback
    return mode, cleaned

In [ ]:
YESNO_START = re.compile(r"^(is|are|was|were|do|does|did|can|could|will|would|should|may|might|must)\b", re.I)

def detect_mode(query: str):
    q = query.lower().strip()

    # ✅ PRIORITY: jurisprudence-style questions should NOT be forced into YESNO
    juris_triggers = ["people v", "g.r.", "gr no", "supreme court", "ruling", "held", "doctrine", "case"]
    if any(t in q for t in juris_triggers):
        # If user asks for quote/exact wording, honor that
        if any(x in q for x in ["quote", "exact wording", "verbatim"]):
            return "QUOTES"
        # Otherwise give a short case-digest style answer
        return "DETAILED"

    # explicit request
    if "yes or no" in q or "yes/no" in q:
        return "YESNO"
    if YESNO_START.match(q):
        return "YESNO"

    # quotes/citations
    if any(x in q for x in ["quote", "exact wording", "verbatim", "pinpoint", "where does it say"]):
        return "QUOTES"
    if any(x in q for x in ["show sources", "sources only", "citations only", "cite only", "provide citations"]):
        return "CITATIONS"

    # main modes
    if any(x in q for x in ["summarize", "summary", "tldr", "in short"]):
        return "SUMMARY"
    if any(x in q for x in ["explain in detail", "elaborate", "expand", "lengthen", "deep dive"]):
        return "DETAILED"
    if any(x in q for x in ["list", "enumerate", "what are", "give me all", "provide a list"]):
        return "LIST"
    if any(x in q for x in ["steps", "step-by-step", "procedure", "process", "how do i", "how to"]):
        return "STEPS"
    if any(x in q for x in ["compare", "difference", "vs", "versus"]):
        return "COMPARE"
    if any(x in q for x in ["what if", "suppose", "in this scenario", "situation", "if someone"]):
        return "SCENARIO"

    return "DEFAULT"

# Coref & Rewrite

## Coref

In [ ]:
chat_history = []
MAX_HISTORY_TURNS = 4

In [ ]:
PRON = re.compile(r"\b(it|this|that|they|them|those|these|the act|the law)\b", re.I)

def _pick_representative(cluster_texts):
    # Prefer a longer, contentful mention (avoid pure pronouns)
    best = ""
    for m in cluster_texts:
        m2 = m.strip()
        if not m2:
            continue
        if PRON.fullmatch(m2):
            continue
        if len(m2) > len(best):
            best = m2
    return best or cluster_texts[0].strip()

def resolve_query_with_fastcoref(coref_model, query: str) -> str:
    if not PRON.search(query):
        return query

    pred = coref_model.predict(texts=[query])[0]
    clusters_str = pred.get_clusters()  # list[list[str]] :contentReference[oaicite:5]{index=5}

    # Map pronoun mentions -> representative string
    repl = {}
    for cl in clusters_str:
        rep = _pick_representative(cl)
        for m in cl:
            if PRON.fullmatch(m.strip() or ""):
                repl[m.strip()] = rep

    # Apply replacements (word-boundary-ish)
    out = query
    # replace longer keys first (e.g., "the law" before "it")
    for k in sorted(repl.keys(), key=len, reverse=True):
        out = re.sub(rf"\b{re.escape(k)}\b", repl[k], out, flags=re.I)
    return out

def build_history_window(chat_history, max_turns=4):
    recent = chat_history[-max_turns*2:]
    parts = []
    for m in recent:
        role = m.get("role","").upper()
        content = (m.get("content","") or "").strip()
        if content:
            parts.append(f"{role}: {content}")
    return "\n".join(parts)

def maybe_coref_enrich_query_fastcoref(coref_model, chat_history, q: str) -> str:
    if not PRON.search(q):
        return q

    window = build_history_window(chat_history, MAX_HISTORY_TURNS)
    combined = (window + "\nUSER: " + q).strip()

    pred = coref_model.predict(texts=[combined])[0]
    # easiest: use string clusters, replace in the combined text
    clusters = pred.get_clusters()  # :contentReference[oaicite:6]{index=6}

    # Build replacements for pronouns anywhere in combined text
    repl = {}
    for cl in clusters:
        rep = _pick_representative(cl)
        for m in cl:
            m2 = m.strip()
            if PRON.fullmatch(m2 or ""):
                repl[m2] = rep

    resolved = combined
    for k in sorted(repl.keys(), key=len, reverse=True):
        resolved = re.sub(rf"\b{re.escape(k)}\b", repl[k], resolved, flags=re.I)

    m = re.search(r"\bUSER:\s*(.*)$", resolved, re.I | re.M)
    return (m.group(1).strip() if m else q)


## Normalization

In [ ]:
def normalize_for_retrieval(q: str) -> str:
    """
    Lightweight coref + intent normalization to improve retrieval.
    - Replaces vague pronouns with RA 9165 when likely.
    - Maps "consequences" -> "penalties/sanctions" wording (legal PDF vocabulary).
    """
    q2 = q.strip()

    if re.search(r"\b(it|this|that|the act|the law)\b", q2, flags=re.I):
        q2 = re.sub(r"\b(it|this|that|the act|the law)\b",
                    "RA 9165 (Dangerous Drugs Act of 2002)", q2, flags=re.I)

    # Map casual wording to legal terms used in PDFs
    q2 = re.sub(r"\bconsequence(s)?\b", "penalties and sanctions", q2, flags=re.I)
    q2 = re.sub(r"\bdisobey(ing)?\b", "violation", q2, flags=re.I)

    return q2

In [ ]:
def normalize_for_retrieval_v2(q: str) -> str:
    q2 = q.strip()

    # Only inject RA 9165 if the user mentions it or the phrase "dangerous drugs act"
    # or the conversation is clearly about RA 9165 via chat history.
    mentions_ra = re.search(r"\b(ra\s*9165|dangerous\s+drugs\s+act)\b", q2, re.I) is not None
    hist_text = " ".join([m.get("content","") for m in chat_history[-MAX_HISTORY_TURNS*2:]]).lower()
    hist_about_ra = ("ra 9165" in hist_text) or ("dangerous drugs act" in hist_text)

    # Replace vague pronouns ONLY if we're confident the topic is RA9165.
    if (mentions_ra or hist_about_ra) and re.search(r"\b(it|this|that|the act|the law)\b", q2, re.I):
        q2 = re.sub(r"\b(it|this|that|the act|the law)\b",
                    "RA 9165 (Dangerous Drugs Act of 2002)", q2, flags=re.I)

    # Light vocab mapping (safe)
    q2 = re.sub(r"\bconsequence(s)?\b", "penalties and sanctions", q2, flags=re.I)
    q2 = re.sub(r"\bdisobey(ing)?\b", "violation", q2, flags=re.I)
    return q2


## Rewrite

In [ ]:
def rewrite_query_for_retrieval(user_query: str) -> str:
    """
    Rewrite a follow-up question into a standalone search query using recent chat history.
    Output MUST be a single line. Do not answer the question.
    Works correctly with gen(..., return_full_text=False).
    """
    recent = chat_history[-MAX_HISTORY_TURNS * 2:]  # user+assistant messages

    messages = [
        {
            "role": "system",
            "content": (
                "Rewrite follow-up questions into standalone search queries for RA 9165 (Dangerous Drugs Act of 2002). "
                "Do NOT answer. Output ONLY ONE LINE. "
                "Replace pronouns like it/this/that/the act/the law with 'RA 9165 (Dangerous Drugs Act of 2002)'. "
                "If the user asks about consequences, rewrite using legal terms like 'penalties', 'sanctions', and 'criminal liability'. "
                "Do not add facts not present in the conversation."
            )
        },
        {
            "role": "user",
            "content": f"""
Conversation (most recent last):
{recent}

Task:
Rewrite the user's question into a standalone query that includes the missing context.
Rules:
- Output exactly ONE line.
- Do NOT answer the question.
- Do NOT add facts not present in the conversation.
- If already standalone, return it unchanged.

User question: {user_query}
Standalone retrieval query:
""".strip()
        }
    ]

    prompt_text = chat_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )

    # With return_full_text=False, generated_text is ONLY the new text
    out = gen(prompt_text)[0]["generated_text"].strip()

    # sanitize to 1 line
    rewritten = " ".join(out.split())

    # safety fallback
    if not rewritten:
        rewritten = " ".join(user_query.strip().split())

    return rewritten


# Retrieval

In [ ]:
def retrieve_final_hits(user_query: str, k: int = 20):
    mode_override, cleaned_query = parse_control_block(user_query)
    mode = mode_override or detect_mode(cleaned_query)

    normalized = normalize_for_retrieval_v2(cleaned_query)
    normalized2 = maybe_coref_enrich_query_fastcoref(coref_model, chat_history, normalized)

    if len(normalized2.split()) < 6 or re.search(r"\b(it|this|that|the act|the law|those|them)\b", normalized2, re.I):
        retrieval_query = rewrite_query_for_retrieval(normalized2)
    else:
        retrieval_query = normalized2

    # Detect if question spans both jurisprudence and statutes
    q_lower = cleaned_query.lower()
    needs_juris = any(x in q_lower for x in ["people v", "ruling", "case", "lim", "consada", "sarabia", "padollo", "asaytuno"])
    needs_statutes = any(x in q_lower for x in ["ra 9165", "ra 10640", "section 21", "irr", "ddb", "pdea", "section", "article"])

    corpora_to_search = []
    if needs_juris:
        corpora_to_search.append("jurisprudence_structured")
    if needs_statutes:
        corpora_to_search.append("statutes_and_guidelines")
    if not corpora_to_search:
        corpora_to_search = [route_corpus(cleaned_query)]

    all_dense, all_sparse = [], []

    for corpus_name in corpora_to_search:
        active_store = stores[corpus_name]
        # temporarily swap globals
        global chunks, embeddings, index, bm25
        chunks = active_store["chunks"]
        embeddings = active_store["embeddings"]
        index = active_store["index"]
        bm25 = active_store["bm25"]

        cand_k = min(120, len(chunks))
        all_dense.extend(retrieve_hnsw(retrieval_query, k=cand_k))
        all_sparse.extend(retrieve_bm25(retrieval_query, k=cand_k))

    # fuse across corpora, rerank, diversify
    fused_hits = rrf_fusion_weighted(all_dense, all_sparse, rrf_k=60, w_dense=0.9, w_sparse=1.3)
    fused_hits = rerank_hits(retrieval_query, fused_hits, top_n=min(100, len(fused_hits)))
    fused_hits = diversify_by_page(fused_hits, max_per_page=2)

    mmr_k = min(len(fused_hits), k * 3)
    mmr_hits = select_diverse_hits_v2(fused_hits, top_k=mmr_k, lambda_mult=0.92)
    mmr_hits = sorted(mmr_hits, key=lambda h: float(h.get("rerank_score", h.get("score", 0.0)) or 0.0), reverse=True)

    return mmr_hits[:k]

# Response

## Dynamic K

In [ ]:
def dynamic_k(mode: str):
    if mode in ["SUMMARY"]:
        return {"cand": 35, "rerank": 20, "final": 6}
    if mode in ["YESNO", "QUOTES", "CITATIONS"]:
        return {"cand": 45, "rerank": 25, "final": 7}
    if mode in ["LIST", "STEPS"]:
        return {"cand": 55, "rerank": 30, "final": 9}
    if mode in ["COMPARE", "SCENARIO", "DETAILED"]:
        return {"cand": 70, "rerank": 40, "final": 10}
    return {"cand": 45, "rerank": 25, "final": 7}

## Build

In [ ]:
def build_system_prompt():
    return (
        "You are a STRICT legal QA assistant for Republic Act No. 9165 and related Philippine jurisprudence.\n"
        "\n"
        "NON-NEGOTIABLE RULES:\n"
        "1) Use ONLY the provided context passages. Do NOT use outside knowledge.\n"
        "2) Answer ONLY the user's question. Do NOT add extra background, history, or general legal discussion.\n"
        "3) Every paragraph or bullet that states facts MUST include at least one citation like [1].\n"
        "4) Do NOT invent citations. Only cite the passage numbers that appear in the context.\n"
        "5) If the context does not clearly support the answer, you MUST say:\n"
        "   I don't know based on the provided PDFs.\n"
        "   and provide: CITATIONS: none\n"
        "6) Prefer quoting the context (short excerpts) when asked what a case/law 'says'.\n"
        "7) Be clear and human-readable, but NEVER sacrifice accuracy for readability.\n"
        "\n"
        "STYLE:\n"
        "- Plain English.\n"
        "- No flowery writing.\n"
        "- No speculation.\n"
    )

import re

IDK_LINE = "I don't know based on the provided PDFs."

def build_onepass_prompt(mode: str, context: str, question: str) -> str:
    """
    FIXED:
    - Removes the contradictory "output exactly ..." rule.
    - If insufficient, model must still output the SAME strict format:
        CITATIONS: none
        ANSWER:
        I don't know...
    """
    mode_rules = {
        "SUMMARY":   "Write 3–6 bullets. Each bullet ends with [n].",
        "DETAILED":  "Write a structured answer with headings. Cite at end of each section [n].",
        "LIST":      "Write bullet points only. Each bullet MUST contain at least 8 words BEFORE the citation, then end with [n].",
        "STEPS":     "Write numbered steps. Each step ends with [n].",
        "COMPARE":   "Compare clearly (A vs B). Cite each comparison block [n].",
        "SCENARIO":  "Use IRAC headings. Cite Rule and Application parts [n].",
        "YESNO":   "Start with: Answer: Yes/No/Not determinable from context. Then 1 short paragraph with citations at the end like [n].",
        "CITATIONS": "Output ONLY a list of sources with citation numbers and (source, page). No extra text.",
        "QUOTES":    "Provide up to 2 short quotes (<=25 words each) and explain briefly. Each quote must have [n].",
        "DEFAULT": "Answer in 1–2 short paragraphs. Put citations at the end of each paragraph like [1]."
    }[mode]

    return f"""
You are a careful legal assistant.

Use ONLY the provided context. Do NOT use outside knowledge.
Every claim must be supported by the context.

If the context is insufficient to answer the question, you MUST output exactly this format:
ANSWER:
{IDK_LINE}

CITATIONS: none

Otherwise, you MUST produce BOTH:
1) CITATIONS: the list of citation numbers you actually used (e.g., [1], [3]) OR 'none'
2) ANSWER: the final answer following the mode rules.

Mode: {mode}
Mode rules:
{mode_rules}

OUTPUT FORMAT EXACTLY:
ANSWER:
<your answer here>

CITATIONS: [n], [n], ...

Context (each passage labeled with [n]):
{context}

Question:
{question}
""".strip()


def normalize_onepass_output(text: str) -> str:
    """
    Ensures output always contains:
      CITATIONS: ...
      ANSWER:
      ...
    Handles these common model failures:
    - "I don't know..." alone
    - "CITATIONS: I don't know..."
    - missing ANSWER block
    """
    t = (text or "").strip()
    if not t:
        return "CITATIONS: none\nANSWER:\n" + IDK_LINE

    # If model returned only the IDK line (or starts with it), wrap it
    if t == IDK_LINE or t.startswith(IDK_LINE):
        return "CITATIONS: none\nANSWER:\n" + IDK_LINE

    # If it mistakenly put IDK under CITATIONS:
    # e.g. "CITATIONS: I don't know based on the provided PDFs."
    if re.match(r"^CITATIONS:\s*I don't know based on the provided PDFs\.?\s*$", t, flags=re.I):
        return "CITATIONS: none\nANSWER:\n" + IDK_LINE

    # If missing CITATIONS header
    if not re.search(r"^CITATIONS:", t, flags=re.M):
        # try to see if it has ANSWER:
        if re.search(r"^ANSWER:", t, flags=re.M):
            return "CITATIONS: none\n" + t
        # otherwise wrap everything as answer
        return "CITATIONS: none\nANSWER:\n" + t

    # If has CITATIONS but missing ANSWER:
    if not re.search(r"^ANSWER:", t, flags=re.M):
        # split after CITATIONS line
        lines = t.splitlines()
        if lines:
            cit_line = lines[0].strip()
            rest = "\n".join(lines[1:]).strip()
            if not rest:
                rest = IDK_LINE
            return cit_line + "\nANSWER:\n" + rest
        return "CITATIONS: none\nANSWER:\n" + IDK_LINE

    return t



## Answer

In [ ]:
def answer(query: str, k=300):
    global chunks, embeddings, index, bm25

    # ---------------------------------------------------------
    # A) Control block tags
    # ---------------------------------------------------------
    mode_override, cleaned_query = parse_control_block(query)

    # ---------------------------------------------------------
    # B) Detect mode
    # ---------------------------------------------------------
    mode = mode_override or detect_mode(cleaned_query)

    # ---------------------------------------------------------
    # C) Normalize + rewrite
    # ---------------------------------------------------------
    normalized = normalize_for_retrieval_v2(cleaned_query)

    # ✅ Minimal retrieval query (no topic-layer dependencies)
    normalized2 = maybe_coref_enrich_query_fastcoref(coref_model, chat_history, normalized)

    # if still short/vague, rewrite to standalone
    if len(normalized2.split()) < 6 or re.search(r"\b(it|this|that|the act|the law|the bill|those|them)\b", normalized2, re.I):
        retrieval_query = rewrite_query_for_retrieval(normalized2)
    else:
        retrieval_query = normalized2

    # ---------------------------------------------------------
    # D) Route corpus
    # ---------------------------------------------------------
    chosen = route_corpus(cleaned_query)
    active_store = stores[chosen]

    chunks = active_store["chunks"]
    embeddings = active_store["embeddings"]
    index = active_store["index"]
    bm25 = active_store["bm25"]

    # ---------------------------------------------------------
    # E) Dynamic k
    # ---------------------------------------------------------
    # --- in answer(), right before retrieval ---
    ks = dynamic_k(mode)

    cand_k   = min(max(ks["cand"], 120), len(chunks))
    rerank_k = min(max(ks["rerank"], 60), len(chunks))
    final_k = min(ks["final"], 12)

    dense_hits  = retrieve_hnsw(retrieval_query, k=cand_k)
    sparse_hits = retrieve_bm25(retrieval_query, k=cand_k)

    # Hybrid fusion + rerank
    fused_hits = rrf_fusion(dense_hits, sparse_hits, rrf_k=60)
    fused_hits = rerank_hits(retrieval_query, fused_hits, top_n=rerank_k)

    fused_hits = diversify_by_page(fused_hits, max_per_page=1)

    mmr_k = min(len(fused_hits), max(final_k * 3, final_k))
    mmr_hits = select_diverse_hits_v2(fused_hits, top_k=mmr_k, lambda_mult=0.85)

    mmr_hits = sorted(
        mmr_hits,
        key=lambda h: float(h.get("rerank_score", h.get("score", 0.0)) or 0.0),
        reverse=True
    )
    hits = mmr_hits[:final_k]


    if not hits:
        reply = "CITATIONS: none\nANSWER:\n" + IDK_LINE
        suggestions = suggest_queries(cleaned_query)
        reply2 = reply + "\n\nTry asking one of these:\n" + "\n".join([f"- {s}" for s in suggestions])
        chat_history.append({"role": "user", "content": query})
        chat_history.append({"role": "assistant", "content": reply2})
        chat_history[:] = chat_history[-MAX_HISTORY_TURNS * 2:]
        return reply2, hits

    context = "\n\n".join(
        [f"[{i+1}] (Source: {h['source']}, page {h['page']})\n{h['text']}"
         for i, h in enumerate(hits)]
    )

    sys = build_system_prompt()
    onepass_msgs = [
        {"role": "system", "content": sys},
        {"role": "user", "content": build_onepass_prompt(mode, context, cleaned_query)}
    ]
    onepass_prompt = chat_tokenizer.apply_chat_template(onepass_msgs, add_generation_prompt=True, tokenize=False)

    onepass_out = gen(onepass_prompt)[0]["generated_text"]
    reply = normalize_onepass_output((onepass_out or "").strip())

    m = re.search(r"^ANSWER:\s*(.*)$", reply, re.M | re.S)
    answer_body = (m.group(1).strip() if m else "")

    if has_uncited_sentences(answer_body):
        reply = regenerate_strictly(mode, context, cleaned_query)

    chat_history.append({"role": "user", "content": query})
    chat_history.append({"role": "assistant", "content": reply})
    chat_history[:] = chat_history[-MAX_HISTORY_TURNS * 2:]

    return reply, hits


# Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Output area
out = widgets.Output(layout=widgets.Layout(
    padding="10px",
    width="100%",
    height="420px",
    overflow_y="auto"
))

# Input + button
inp = widgets.Text(
    placeholder="Ask a question about RA 9165…",
    description="You:",
    layout=widgets.Layout(width="75%")
)

btn = widgets.Button(
    description="Send",
    button_style="primary"
)

# Status line (for Thinking...)
status = widgets.HTML(value="", layout=widgets.Layout(margin="6px 0 0 0"))

def show_thinking():
    # Animated dots using CSS (no JS needed)
    status.value = """
    <div style="font-family: monospace; font-size: 14px;">
      Thinking<span class="dots"></span>
    </div>
    <style>
      .dots::after {
        content: '';
        display: inline-block;
        width: 1.5em;
        text-align: left;
        animation: dots 1s steps(4, end) infinite;
      }
      @keyframes dots {
        0%   { content: ''; }
        25%  { content: '.'; }
        50%  { content: '..'; }
        75%  { content: '...'; }
        100% { content: ''; }
      }
    </style>
    """

def clear_thinking():
    status.value = ""

def set_busy(is_busy: bool):
    inp.disabled = is_busy
    btn.disabled = is_busy

def on_submit(_=None):
    q = inp.value.strip()
    if not q:
        return

    inp.value = ""
    set_busy(True)
    show_thinking()

    try:
        resp, hits = answer(q, k=25)
        with out:
            print(f"You: {q}\n")
            print(f"Assistant:")
            print(f"{resp}\n")
            # print("Sources:")
            # for i, h in enumerate(hits, 1):
            #     rrf_sc = float(h.get("score", 0.0))
            #     ce_sc  = h.get("rerank_score", None)
            #     if ce_sc is None:
            #         print(f"  [{i}] {h['source']} p.{h['page']}  rrf={rrf_sc:.3f}")
            #     else:
            #         print(f"  [{i}] {h['source']} p.{h['page']}  ce={float(ce_sc):.3f}  rrf={rrf_sc:.3f}")
            # print("\n" + "-" * 80 + "\n")
    finally:
        clear_thinking()
        set_busy(False)

inp.continuous_update = False
inp.on_submit(lambda _: on_submit())
btn.on_click(on_submit)

display(widgets.VBox([
    widgets.HBox([inp, btn]),
    status,
    out
]))


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import json

# -----------------------------
# Output area
# -----------------------------
out = widgets.Output(layout=widgets.Layout(
    padding="10px",
    width="100%",
    height="420px",
    overflow_y="auto"
))

# -----------------------------
# Input + button
# -----------------------------
inp = widgets.Text(
    placeholder="Ask a question about RA 9165…",
    description="You:",
    layout=widgets.Layout(width="75%")
)

btn = widgets.Button(
    description="Send",
    button_style="primary"
)

# -----------------------------
# Status line (for Thinking...)
# -----------------------------
status = widgets.HTML(value="", layout=widgets.Layout(margin="6px 0 0 0"))

def show_thinking():
    # Animated dots using CSS (no JS needed)
    status.value = """
    <div style="font-family: monospace; font-size: 14px;">
      Thinking<span class="dots"></span>
    </div>
    <style>
      .dots::after {
        content: '';
        display: inline-block;
        width: 1.5em;
        text-align: left;
        animation: dots 1s steps(4, end) infinite;
      }
      @keyframes dots {
        0%   { content: ''; }
        25%  { content: '.'; }
        50%  { content: '..'; }
        75%  { content: '...'; }
        100% { content: ''; }
      }
    </style>
    """

def clear_thinking():
    status.value = ""

def set_busy(is_busy: bool):
    inp.disabled = is_busy
    btn.disabled = is_busy

# -----------------------------
# Context helpers (JSON-style)
# -----------------------------
def dedupe_context_texts(hits):
    """Exact-text dedupe while preserving order."""
    seen = set()
    unique = []
    for h in (hits or []):
        t = (h.get("text") or "").strip()
        if not t:
            continue
        if t in seen:
            continue
        seen.add(t)
        unique.append(t)
    return unique

def build_contexts_json(hits, max_items=25):
    """
    Returns a dict like:
      {"contexts": ["...", "...", ...]}
    """
    texts = dedupe_context_texts(hits)
    if max_items is not None:
        texts = texts[:int(max_items)]
    return {"contexts": texts}

# -----------------------------
# Main submit handler
# -----------------------------
def on_submit(_=None):
    q = inp.value.strip()
    if not q:
        return

    inp.value = ""
    set_busy(True)
    # show_thinking()

    try:
        # NOTE: answer(...) must already be defined in your notebook
        resp, hits = answer(q, k=25)

        contexts_json = build_contexts_json(hits, max_items=25)

        with out:
            print(f"You: {q}\n")
            print("Assistant:")
            print(f"{resp}\n")

            # ✅ JSON-style contexts output (like your example)
            print("Retrieved Context (JSON):")
            print(json.dumps(contexts_json, indent=2, ensure_ascii=False))
            print()

            print("Sources:")
            for i, h in enumerate(hits, 1):
                rrf_sc = float(h.get("score", 0.0))
                ce_sc  = h.get("rerank_score", None)
                if ce_sc is None:
                    print(f"  [{i}] {h.get('source')} p.{h.get('page')}  rrf={rrf_sc:.3f}")
                else:
                    print(f"  [{i}] {h.get('source')} p.{h.get('page')}  ce={float(ce_sc):.3f}  rrf={rrf_sc:.3f}")

            print("\n" + "-" * 80 + "\n")

    finally:
        clear_thinking()
        set_busy(False)

# -----------------------------
# Wire up UI
# -----------------------------
inp.continuous_update = False
inp.on_submit(lambda _: on_submit())
btn.on_click(on_submit)

display(widgets.VBox([
    widgets.HBox([inp, btn]),
    status,
    out
]))

# Testing

## Batch Run

In [ ]:
# # Uncomment if proceeding with a batch run for RAGAS

# Question = [

# # ============================================================
# # JURISPRUDENCE (10 — EASY TO AVERAGE)
# # ============================================================

# "What was the main issue in People v. Lim?",
# "What did the Supreme Court say about Section 21 compliance in People v. Lim?",
# "Did the Court acquit or convict in People v. Lim?",
# "What does People v. Lim say about justifiable grounds for noncompliance?",
# "What is the ruling in People v. Sarabia regarding chain of custody?",
# "Did the accused win in People v. Sarabia?",
# "What was the issue in People v. Padollo?",
# "Did People v. Padollo require strict compliance with Section 21?",
# "What was discussed in People v. Asaytuno about handling seized drugs?",
# "What did the Court say about presumption of regularity in drug cases?",


# # ============================================================
# # STATUTES + GUIDELINES (40 — EASY TO AVERAGE)
# # ============================================================

# # -------------------------------
# # RA 9165 (20)
# # -------------------------------

# "What is Section 5 of RA 9165 about?",
# "What is Section 11 of RA 9165 about?",
# "What is Section 21 of RA 9165?",
# "What is meant by 'dangerous drugs' under RA 9165?",
# "What are the penalties for illegal sale of dangerous drugs?",
# "What are the penalties for illegal possession of dangerous drugs?",
# "Who must be present during inventory under Section 21 (original law)?",
# "When should inventory and photographing be done under RA 9165?",
# "Where should inventory be conducted under the original Section 21?",
# "What is the purpose of Section 21 procedures?",
# "What government agency enforces RA 9165?",
# "What does RA 9165 say about confiscation of drugs?",
# "What happens to seized drugs after confiscation?",
# "What is the role of PDEA under RA 9165?",
# "What is the role of the Dangerous Drugs Board under RA 9165?",
# "What is considered illegal sale under RA 9165?",
# "What is considered illegal possession under RA 9165?",
# "Does RA 9165 allow noncompliance with Section 21?",
# "What documents are prepared during inventory of seized drugs?",
# "What is the definition of 'chain of custody'?",


# # -------------------------------
# # RA 10640 (8)
# # -------------------------------

# "What is RA 10640?",
# "What did RA 10640 change in Section 21?",
# "How many witnesses are required under RA 10640?",
# "Did RA 10640 remove the DOJ representative requirement?",
# "Where can inventory be conducted under RA 10640?",
# "What does RA 10640 say about justifiable grounds?",
# "Did RA 10640 reduce the number of required witnesses?",
# "What is the purpose of RA 10640?",


# # -------------------------------
# # DDB IRR (6)
# # -------------------------------

# "What does the DDB IRR say about inventory of seized drugs?",
# "What does the DDB IRR say about photographing seized items?",
# "What is required documentation under the DDB IRR?",
# "Does the DDB IRR mention chain of custody?",
# "Who signs the inventory under the DDB IRR?",
# "What does the DDB IRR say about handling seized drugs?",


# # -------------------------------
# # PDEA Guidelines (4)
# # -------------------------------

# "What are the basic steps in a buy-bust operation under PDEA Guidelines?",
# "Is coordination with PDEA required before a buy-bust?",
# "What do the PDEA Guidelines say about marking seized drugs?",
# "What do the PDEA Guidelines say about submitting drugs to the laboratory?",


# # -------------------------------
# # Rules of Court (2)
# # -------------------------------

# "What is Rule 110 of the Rules of Court about?",
# "How is criminal information filed under the Rules of Court?",

# ]

Batch Run with Ragas JSON Formatter

In [ ]:
# import json, time, os, gc
# from pathlib import Path
# import torch
# import re

# # -----------------------------
# # CONFIG
# # -----------------------------
# OUT_DIR = Path("/content/drive/MyDrive/PGD_Colab/eval")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
# OUT_PATH = OUT_DIR / f"3b_contexts_{RUN_TAG}.json"

# TOP_K = 25          # passed into answer(q, k=TOP_K)
# SAVE_EVERY = 1      # save after every N questions (1 = safest)

# # -----------------------------
# # HELPERS
# # -----------------------------
# def gpu_cleanup():
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

# def atomic_save_json(path: Path, data):
#     tmp = str(path) + ".tmp"
#     with open(tmp, "w", encoding="utf-8") as f:
#         json.dump(data, f, ensure_ascii=False, indent=2)
#     os.replace(tmp, path)

# def clear_history_if_exists():
#     # keep batch independent (prevents rewrite/coref from using previous Q/A)
#     try:
#         chat_history.clear()
#     except Exception:
#         pass

# def extract_answer_text(resp: str) -> str:
#     """
#     Returns the text under ANSWER: ... (up to CITATIONS:).
#     If format is missing, returns resp as-is.
#     """
#     if resp is None:
#         return ""
#     t = str(resp).strip()
#     if not t:
#         return ""

#     m = re.search(r"^ANSWER:\s*(.*)$", t, flags=re.M | re.S)
#     if not m:
#         return t  # fallback: raw response

#     body = (m.group(1) or "").strip()
#     m2 = re.search(r"^CITATIONS:\s*", body, flags=re.M)
#     if m2:
#         body = body[:m2.start()].strip()

#     return body if body else ""  # keep it simple

# def hits_to_contexts(hits, max_items=25):
#     """
#     contexts = list[str] from hits[].text (deduped, preserving order)
#     """
#     seen = set()
#     contexts = []
#     for h in (hits or []):
#         if not isinstance(h, dict):
#             continue
#         txt = (h.get("text") or "").strip()
#         if not txt or txt in seen:
#             continue
#         seen.add(txt)
#         contexts.append(txt)
#         if max_items is not None and len(contexts) >= int(max_items):
#             break
#     return contexts

# # -----------------------------
# # MAIN LOOP
# # -----------------------------
# def simple_batch(Question, out_path=OUT_PATH, k=TOP_K):
#     results = []
#     total = len(Question)

#     for i, q in enumerate(Question, start=1):
#         gpu_cleanup()
#         clear_history_if_exists()

#         # call your existing function (same as UI uses)
#         resp, hits = answer(str(q), k=k)

#         rec = {
#             "question": str(q),
#             "answer": extract_answer_text(resp),
#             "contexts": hits_to_contexts(hits, max_items=k),
#         }
#         results.append(rec)

#         if (i % SAVE_EVERY) == 0:
#             atomic_save_json(out_path, results)

#         print(f"[{i}/{total}] saved → {out_path.name}")

#     atomic_save_json(out_path, results)
#     print(f"\n✅ DONE. Output saved to: {out_path}")
#     return results

# # -----------------------------
# # RUN
# # -----------------------------
# _ = simple_batch(Question)

Retrieval Metrics JSON Formatter

In [ ]:
# import json
# import time
# from pathlib import Path

# # -------------------------------------------------
# # CONFIG
# # -------------------------------------------------
# OUT_DIR = Path("/content/drive/MyDrive/PGD_Colab/eval")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
# OUT_PATH = OUT_DIR / f"ground_truth_pages_{RUN_TAG}.json"

# # -------------------------------------------------
# # BUILD GROUND TRUTH TEMPLATE
# # -------------------------------------------------
# ground_truth_data = []

# for i, q in enumerate(Question, start=1):
#     ground_truth_data.append({
#         "QID": i,
#         "Question": q,
#         "ground_truth": [
#             # Example format:
#             # "People_v_Lim.pdf page 12",
#             # "RA_9165.pdf page 4"
#         ]
#     })

# # -------------------------------------------------
# # SAVE
# # -------------------------------------------------
# with open(OUT_PATH, "w", encoding="utf-8") as f:
#     json.dump(ground_truth_data, f, ensure_ascii=False, indent=2)

# print(f"✅ Ground truth (page-level text format) saved to:\n{OUT_PATH}")
# print(f"Total questions: {len(ground_truth_data)}")

Retrieval Metrics Evaluation

In [ ]:
# import os, json, time, math, re, inspect
# from pathlib import Path

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# # -----------------------------
# # CONFIG (EDIT THESE)
# # -----------------------------
# EVAL_DIR = Path("/content/drive/MyDrive/PGD_Colab/eval")
# GT_PATH  = EVAL_DIR / "ground_truth_pages_20260224_151743.json"   # <-- your ground_truth.json

# KS = [1, 3, 5, 10, 15, 20]
# MAXK = max(KS)

# # retrieve more than MAXK so unique-page de-dup still leaves enough unique pages
# KMAX_RETRIEVE = 80

# RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
# OUT_BATCH = EVAL_DIR / f"retrieval_only_{RUN_TAG}.json"
# OUT_SUM   = EVAL_DIR / f"summary_metrics_vsK_{RUN_TAG}.csv"
# OUT_PERQ  = EVAL_DIR / f"per_qid_metrics_vsK_{RUN_TAG}.csv"

# PLOT_PR   = EVAL_DIR / f"plot_precision_recall_{RUN_TAG}.png"
# PLOT_HIT  = EVAL_DIR / f"plot_hit_{RUN_TAG}.png"
# PLOT_MAP  = EVAL_DIR / f"plot_map_{RUN_TAG}.png"
# PLOT_MRR  = EVAL_DIR / f"plot_mrr_{RUN_TAG}.png"
# PLOT_NDCG = EVAL_DIR / f"plot_ndcg_{RUN_TAG}.png"

# EVAL_DIR.mkdir(parents=True, exist_ok=True)

# # -----------------------------
# # SAFETY CHECK: you must already have retrieve_final_hits() defined
# # -----------------------------
# if "retrieve_final_hits" not in globals():
#     raise RuntimeError("retrieve_final_hits() is not defined. Run your pipeline cell that defines it first.")

# # -----------------------------
# # LOAD GROUND TRUTH
# # -----------------------------
# with open(GT_PATH, "r", encoding="utf-8") as f:
#     gt_rows = json.load(f)

# gt_by_qid = {}
# for r in gt_rows:
#     if "QID" not in r or "Question" not in r:
#         continue
#     gt_by_qid[int(r["QID"])] = r

# qids = sorted(gt_by_qid.keys())
# print(f"✅ Loaded ground truth rows: {len(qids)} from: {GT_PATH.name}")

# # -----------------------------
# # Normalization helpers
# # -----------------------------
# def _bn(p: str) -> str:
#     """basename, lower, stripped; supports full paths"""
#     return Path(str(p)).name.strip().lower()

# def _try_int(x):
#     try:
#         return int(x)
#     except Exception:
#         return None

# def page_key(source: str, page: int):
#     return (_bn(source), int(page))

# # -----------------------------
# # DOC-ID normalization (fixes txt vs pdf, structured suffixes, etc.)
# # -----------------------------
# _SUFFIXES_TO_STRIP = set([
#     "structured", "structure", "clean", "cleaned", "chunks", "chunked",
#     "ocr", "scanned", "parsed", "text", "final", "draft", "ver", "version",
#     "v1", "v2", "v3", "copy", "copies"
# ])

# def norm_doc_id(path_or_name: str) -> str:
#     """
#     Convert filename/path into comparable doc-id:
#     - take stem (no extension)
#     - lowercase
#     - split by _ - space
#     - strip trailing suffix tokens repeatedly
#     - remove non-alphanumeric
#     """
#     s = str(path_or_name).replace("\\", "/").strip()
#     stem = Path(s).stem.lower().strip()

#     toks = re.split(r"[_\-\s]+", stem)
#     while toks and toks[-1] in _SUFFIXES_TO_STRIP:
#         toks.pop()

#     joined = "".join(toks)
#     joined = re.sub(r"[^a-z0-9]+", "", joined)
#     return joined

# # -----------------------------
# # Ground truth parsing:
# # - "Some.pdf page 12" => page-level
# # - otherwise => file-level doc-id relevance (txt/pdf variants match)
# # -----------------------------
# _PDF_PAGE_RE = re.compile(r"^(?P<fname>.+?\.pdf)\s+page\s+(?P<page>\d+)\s*$", re.I)

# def parse_gt_list(items):
#     """
#     Returns:
#       gt_page_set: set[(basename_lower, page_int)] for page-level items
#       gt_file_docids: set[doc_id] for file-level items (.txt/.pdf without page)
#     """
#     gt_page_set = set()
#     gt_file_docids = set()

#     for it in (items or []):
#         s = (it or "").strip()
#         if not s:
#             continue

#         s_norm = s.replace("\\", "/").strip()
#         m = _PDF_PAGE_RE.match(s_norm)
#         if m:
#             gt_page_set.add((_bn(m.group("fname")), int(m.group("page"))))
#         else:
#             gt_file_docids.add(norm_doc_id(_bn(s_norm)))

#     return gt_page_set, gt_file_docids

# def is_relevant(pred_bn: str, pred_page: int, gt_page_set, gt_file_docids):
#     """
#     pred_bn should already be basename-lower of the retrieved source
#     Relevant if:
#       - exact (basename,page) matches GT page set
#       - OR doc_id(pred_bn) matches any GT file doc-id
#     """
#     bn = _bn(pred_bn)
#     pg = int(pred_page)

#     if (bn, pg) in gt_page_set:
#         return True

#     if norm_doc_id(bn) in gt_file_docids:
#         return True

#     return False

# # -----------------------------
# # Convert hits -> ranked UNIQUE pages (keeps order; removes duplicates)
# # -----------------------------
# def hits_to_ranked_unique_pages(hits, kmax=None):
#     seen = set()
#     ranked = []
#     for h in (hits or []):
#         src = h.get("source")
#         pg  = h.get("page")
#         if not src:
#             continue
#         pg_i = _try_int(pg)
#         if pg_i is None:
#             continue

#         key = page_key(src, pg_i)
#         if key in seen:
#             continue
#         seen.add(key)
#         ranked.append(key)
#         if kmax and len(ranked) >= int(kmax):
#             break
#     return ranked

# # -----------------------------
# # Metrics @K (binary relevance)
# # -----------------------------
# def precision_at_k(rel_flags, k):
#     return float(np.sum(rel_flags[:k])) / float(k) if k > 0 else 0.0

# def recall_at_k(rel_flags, num_rel, k):
#     return float(np.sum(rel_flags[:k])) / float(num_rel) if num_rel > 0 else 0.0

# def hit_at_k(rel_flags, k):
#     return 1.0 if np.any(rel_flags[:k]) else 0.0

# def mrr_at_k(rel_flags, k):
#     for i in range(min(k, len(rel_flags))):
#         if rel_flags[i]:
#             return 1.0 / float(i + 1)
#     return 0.0

# def ap_at_k(rel_flags, num_rel, k):
#     if num_rel <= 0:
#         return 0.0
#     s = 0.0
#     hits = 0
#     for i in range(1, k + 1):
#         if i-1 < len(rel_flags) and rel_flags[i-1]:
#             hits += 1
#             s += hits / float(i)
#     denom = float(min(num_rel, k))
#     return s / denom if denom > 0 else 0.0

# def ndcg_at_k(rel_flags, num_rel, k):
#     if num_rel <= 0 or k <= 0:
#         return 0.0
#     dcg = 0.0
#     for i in range(1, k + 1):
#         if i-1 < len(rel_flags) and rel_flags[i-1]:
#             dcg += 1.0 / math.log2(i + 1)
#     ideal = min(int(num_rel), int(k))
#     idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal + 1))
#     return (dcg / idcg) if idcg > 0 else 0.0

# # -----------------------------
# # Retrieval call wrapper:
# # - tries to pass k if retrieve_final_hits supports it
# # - counts calls to prove it's ONCE per question
# # -----------------------------
# RETRIEVAL_CALLS = 0

# def call_retrieve_final_hits(question: str, k: int):
#     global RETRIEVAL_CALLS
#     RETRIEVAL_CALLS += 1
#     fn = retrieve_final_hits
#     try:
#         sig = inspect.signature(fn)
#         if "k" in sig.parameters:
#             return fn(question, k=k)
#         if "top_k" in sig.parameters:
#             return fn(question, top_k=k)
#         if "K" in sig.parameters:
#             return fn(question, K=k)
#         return fn(question)
#     except Exception:
#         return fn(question)

# # ============================================================
# # 1) RETRIEVAL-ONLY RUN (ONCE PER QUESTION)
# # ============================================================
# results = []
# for n, qid in enumerate(qids, start=1):
#     q = gt_by_qid[qid]["Question"]

#     rec = {
#         "QID": int(qid),
#         "Question": q,
#         "Retrieved_pages": [],   # raw ranked hits; de-dup at eval
#         "Error": None
#     }

#     try:
#         hits = call_retrieve_final_hits(q, k=KMAX_RETRIEVE)

#         pages = []
#         for h in (hits or []):
#             src = h.get("source")
#             pg  = h.get("page")
#             if src and pg is not None:
#                 pg_i = _try_int(pg)
#                 if pg_i is None:
#                     continue
#                 pages.append({"source": src, "page": int(pg_i)})
#         rec["Retrieved_pages"] = pages

#     except Exception as e:
#         rec["Error"] = f"{type(e).__name__}: {str(e)}"

#     results.append(rec)

#     # incremental save
#     with open(OUT_BATCH, "w", encoding="utf-8") as f:
#         json.dump(results, f, ensure_ascii=False, indent=2)

#     print(f"[{n}/{len(qids)}] saved → {OUT_BATCH.name}" + (" (ERROR)" if rec["Error"] else ""))

# print(f"\n✅ Retrieval-only JSON saved: {OUT_BATCH}")
# print(f"✅ Sanity check: retrieve_final_hits calls = {RETRIEVAL_CALLS} (expected {len(qids)})")

# # ============================================================
# # 2) EVALUATE METRICS @K (from the SAME ranked list)
# # ============================================================
# per_rows = []
# macro = {k: {"P": [], "R": [], "Hit": [], "MAP": [], "MRR": [], "nDCG": []} for k in KS}

# for rec in results:
#     qid = int(rec["QID"])
#     gt_list = gt_by_qid[qid].get("ground_truth", []) or []

#     gt_page_set, gt_file_docids = parse_gt_list(gt_list)

#     # total relevant items (mixed): page-items + file-items
#     num_rel = len(gt_page_set) + len(gt_file_docids)

#     # predicted ranked unique pages up to MAXK
#     pred_ranked = hits_to_ranked_unique_pages(rec.get("Retrieved_pages") or [], kmax=MAXK)

#     # relevance flags length MAXK
#     rel_flags = []
#     for i in range(MAXK):
#         if i < len(pred_ranked):
#             bn, pg = pred_ranked[i]
#             rel_flags.append(1 if is_relevant(bn, pg, gt_page_set, gt_file_docids) else 0)
#         else:
#             rel_flags.append(0)

#     row = {
#         "QID": qid,
#         "num_relevant": num_rel,
#         "num_gt_pages": len(gt_page_set),
#         "num_gt_files": len(gt_file_docids),
#         "num_predicted_unique_pages": len(pred_ranked),
#         "has_error": 1 if rec.get("Error") else 0
#     }

#     for k in KS:
#         P = precision_at_k(rel_flags, k)
#         R = recall_at_k(rel_flags, num_rel, k)
#         H = hit_at_k(rel_flags, k)
#         MAP = ap_at_k(rel_flags, num_rel, k)
#         MRR = mrr_at_k(rel_flags, k)
#         N = ndcg_at_k(rel_flags, num_rel, k)

#         row[f"P@{k}"] = P
#         row[f"R@{k}"] = R
#         row[f"Hit@{k}"] = H
#         row[f"MAP@{k}"] = MAP
#         row[f"MRR@{k}"] = MRR
#         row[f"nDCG@{k}"] = N

#         macro[k]["P"].append(P)
#         macro[k]["R"].append(R)
#         macro[k]["Hit"].append(H)
#         macro[k]["MAP"].append(MAP)
#         macro[k]["MRR"].append(MRR)
#         macro[k]["nDCG"].append(N)

#     per_rows.append(row)

# df_per = pd.DataFrame(per_rows).sort_values("QID").reset_index(drop=True)

# summary_rows = []
# for k in KS:
#     summary_rows.append({
#         "K": k,
#         "Precision": float(np.mean(macro[k]["P"])) if macro[k]["P"] else 0.0,
#         "Recall":    float(np.mean(macro[k]["R"])) if macro[k]["R"] else 0.0,
#         "Hit":       float(np.mean(macro[k]["Hit"])) if macro[k]["Hit"] else 0.0,
#         "MAP":       float(np.mean(macro[k]["MAP"])) if macro[k]["MAP"] else 0.0,
#         "MRR":       float(np.mean(macro[k]["MRR"])) if macro[k]["MRR"] else 0.0,
#         "nDCG":      float(np.mean(macro[k]["nDCG"])) if macro[k]["nDCG"] else 0.0,
#         "num_questions": int(len(df_per)),
#         "num_questions_with_error": int(df_per["has_error"].sum())
#     })
# df_sum = pd.DataFrame(summary_rows)

# # Save CSVs
# df_per.to_csv(OUT_PERQ, index=False, encoding="utf-8")
# df_sum.to_csv(OUT_SUM, index=False, encoding="utf-8")

# print(f"\n✅ Saved per-question CSV: {OUT_PERQ.name}")
# print(f"✅ Saved summary CSV:      {OUT_SUM.name}")

# display(df_sum)

# # ============================================================
# # 3) PLOTS (save PNG + show)
# # ============================================================
# import numpy as np
# import matplotlib.pyplot as plt

# # ============================================================
# # Precision & Recall Plot (Even X-axis 1 → 20)
# # ============================================================
# # ============================================================
# # Precision & Recall Plot
# # ============================================================
# def plot_pr(df, outpath):
#     x = df["K"].tolist()
#     precision = df["Precision"].tolist()
#     recall = df["Recall"].tolist()

#     plt.figure(figsize=(8, 5))
#     plt.plot(x, precision, marker="o", linewidth=2, label="Precision")
#     plt.plot(x, recall, marker="o", linewidth=2, label="Recall")

#     # ---- YOUR REQUESTED AXIS SETTINGS ----
#     plt.ylim(0, 1)
#     plt.xlim(0, 21)
#     plt.xticks(np.arange(2.5, 20.1, 2.5))

#     # Clean larger grid
#     plt.grid(True, linestyle="--", linewidth=0.8)

#     plt.xlabel("K")
#     plt.ylabel("Score")
#     plt.title("Precision@K & Recall@K")
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(outpath, dpi=150)
#     plt.show()


# # ============================================================
# # Generic Metric Plot (Hit, MAP, MRR, nDCG)
# # ============================================================
# def plot_metric(df, col, title, outpath):
#     x = df["K"].tolist()
#     y = df[col].tolist()

#     plt.figure(figsize=(8, 5))
#     plt.plot(x, y, marker="o", linewidth=2)

#     # ---- YOUR REQUESTED AXIS SETTINGS ----
#     plt.ylim(0, 1)
#     plt.xlim(0, 21)
#     plt.xticks(np.arange(2.5, 20.1, 2.5))

#     plt.grid(True, linestyle="--", linewidth=0.8)

#     plt.xlabel("K")
#     plt.ylabel("Score")
#     plt.title(title)
#     plt.tight_layout()
#     plt.savefig(outpath, dpi=150)
#     plt.show()

# plot_pr(df_sum, PLOT_PR)
# plot_metric(df_sum, "Hit",  "Hit@K",  PLOT_HIT)
# plot_metric(df_sum, "MAP",  "MAP@K",  PLOT_MAP)
# plot_metric(df_sum, "MRR",  "MRR@K",  PLOT_MRR)
# plot_metric(df_sum, "nDCG", "nDCG@K", PLOT_NDCG)

# print("\n📌 Outputs written to:", str(EVAL_DIR))
# print(" -", OUT_BATCH.name)
# print(" -", OUT_PERQ.name)
# print(" -", OUT_SUM.name)
# print(" -", PLOT_PR.name)
# print(" -", PLOT_HIT.name)
# print(" -", PLOT_MAP.name)
# print(" -", PLOT_MRR.name)
# print(" -", PLOT_NDCG.name)